### Imports

In [1]:
# Installs (Colab/runtime)
%pip -q install langchain langchain-text-splitters langchain-community bs4 sentence-transformers
%pip -q install -U "langchain[google-genai]"
%pip -q install -U "langchain-core"

import getpass
import os
import re
import shutil
import sys
from pathlib import Path

import pandas as pd
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import CSVLoader
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore

### Google Gemini

In [3]:
os.environ["GOOGLE_API_KEY"] =""
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

model = init_chat_model("google_genai:gemini-2.5-flash")

### Embeddings

In [4]:
# Embeddings: default to LOCAL to avoid Gemini free-tier embed quota.
USE_GEMINI_EMBEDDINGS = False

if USE_GEMINI_EMBEDDINGS:
    from langchain_google_genai import GoogleGenerativeAIEmbeddings

    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
else:
    from langchain_community.embeddings import HuggingFaceEmbeddings

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = InMemoryVectorStore(embeddings)

/tmp/ipython-input-691438408.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


### Data and Chunks

In [6]:
csv_path = Path("fashion.csv")

loader = CSVLoader(file_path=str(csv_path), encoding="utf-8")
docs = loader.load()

all_splits = docs

# Index chunks
_ = vector_store.add_documents(documents=all_splits)

### RAG Agent

In [7]:
# --- RAG chain via dynamic prompt (middleware) ---
# This runs retrieval automatically on every user message and injects the
# retrieved content into the model prompt.
@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    last_msg = request.state["messages"][-1]

    # Be robust across different message object shapes
    last_query = getattr(last_msg, "text", None) or getattr(last_msg, "content", "")

    retrieved_docs = vector_store.similarity_search(last_query, k=10)
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    system_message = (
        "You are a helpful shopping assistant. Use the following product catalog context to answer."
        "If the answer isn't in the context, say you don't know.\n\n"
        f"{docs_content}"
    )

    return system_message

agent = create_agent(model, tools=[], middleware=[prompt_with_context])

### Query

In [8]:
query = "hi, do you have women shoes size 40?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

hi, do you have women shoes size 40?
================================== Ai Message ==================================

Yes, I do! Here are the women's shoes available in size 40:

*   **ADIDAS Women's Vintage Set White Shoe** (ProductId: 5927)
*   **HM Women Blue Flats** (ProductId: 56958)
*   **Catwalk Women Blue Shoes** (ProductId: 46824)
*   **Nike Women Sweet Classic Leather White Shoes** (ProductId: 25490)
*   **HM Women Blue Heels** (ProductId: 56876)
*   **Clarks Women Un Spire White Sandals** (ProductId: 37961)
*   **Numero Uno Women NY Grey Blue Shoe** (ProductId: 4742)
*   **HM Women Blue Heels** (ProductId: 56903)
